# Chapter 1 · 기온·대기질 계절성

**연구질문:** 기온 시계열의 연간 계절성 세기는 얼마이며, 계절 제거 후 잔차는 정상에 가까운가?

**학습 흐름:** 개념 → 자료 → 시각화 → 분해 → 정상성 검정 → 자기상관 → 해석.

> 이 노트북은 위에서 아래로 실행하면 끝까지 동작하도록 만들었습니다. 온라인 자료를 먼저 시도하고, 실패하면 **합성자료로 자동 대체**하며 `DATA_KIND`에 표시합니다.

## 1. 핵심 개념·수식

**약정상성(weak stationarity):** $E(y_t)=\mu$, $\mathrm{Var}(y_t)=\sigma^2$, $\mathrm{Cov}(y_t,y_{t-k})=\gamma(k)$는 $t$에 의존하지 않는다.

**고전적 분해(가법):** $y_t = T_t + S_t + R_t$. 계절성의 세기는 $F_S=\max\!\left(0,\,1-\dfrac{\mathrm{Var}(R_t)}{\mathrm{Var}(S_t+R_t)}\right)$로 요약한다(0에 가까우면 약함).

**단위근 검정:** ADF의 귀무가설은 *단위근 존재*, KPSS의 귀무가설은 *정상성*이다. 두 검정을 함께 읽는다.

| ADF (H0: 단위근) | KPSS (H0: 정상) | 읽는 법 |
|---|---|---|
| 기각 | 기각 못함 | 정상성과 부합 |
| 기각 못함 | 기각 | 단위근/비정상과 부합 → 차분 고려 |
| 둘 다 기각 | | 추세정상·구조변화·분산 불안정 가능 |
| 둘 다 기각 못함 | | 검정력 부족, 판단 보류 |

**자기상관:** $\rho(k)=\gamma(k)/\gamma(0)$. Ljung–Box $Q=n(n+2)\sum_{k=1}^{h}\dfrac{\hat\rho(k)^2}{n-k}$. 변동성 군집은 $r_t^2$의 자기상관(ARCH-LM)으로 점검한다.

In [ ]:
import os, io, json, subprocess, sys, warnings, urllib.request, urllib.parse
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy import stats, optimize

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
SEED = 42
# 네트워크 없이 합성자료로만 실행하려면 환경변수 TSLAB_OFFLINE=1 로 실행하거나 아래를 False로 바꾼다.
USE_LIVE_DATA = os.environ.get("TSLAB_OFFLINE") != "1"
plt.rcParams.update({"figure.figsize": (10, 4), "axes.grid": True,
                     "axes.spines.top": False, "axes.spines.right": False})

def ensure(pkg, mod=None):
    # 없으면 설치를 시도한다. 실패해도 노트북은 계속 실행되고 해당 선택 기능만 건너뛴다.
    try:
        __import__(mod or pkg); return True
    except Exception:
        if not USE_LIVE_DATA:
            return False
        try:
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True, timeout=300)
            __import__(mod or pkg); return True
        except Exception as e:
            print(f"[건너뜀] {pkg} 설치/가져오기 실패: {type(e).__name__}")
            return False

def fetch_bytes(url, timeout=60):
    req = urllib.request.Request(url, headers={"User-Agent": "ts-lab-course/1.0 (education)"})
    with urllib.request.urlopen(req, timeout=timeout) as r:
        return r.read()

# ---- 평가 지표와 기준선 ----
def mae(a, f):  return float(np.mean(np.abs(np.asarray(a, float) - np.asarray(f, float))))
def rmse(a, f): return float(np.sqrt(np.mean((np.asarray(a, float) - np.asarray(f, float)) ** 2)))
def mase(a, f, train, m=1):
    train = np.asarray(train, float)
    scale = np.mean(np.abs(train[m:] - train[:-m]))
    return mae(a, f) / scale
def naive_fc(train, h):  return np.repeat(np.asarray(train, float)[-1], h)
def snaive_fc(train, h, m):
    train = np.asarray(train, float)
    return np.tile(train[-m:], int(np.ceil(h / m)))[:h]
def drift_fc(train, h):
    train = np.asarray(train, float)
    slope = (train[-1] - train[0]) / (len(train) - 1)
    return train[-1] + slope * np.arange(1, h + 1)
def show(df, digits=4):
    print(df.round(digits).to_string())

In [ ]:
def parse_openmeteo_daily(obj):
    d = obj["daily"]; idx = pd.to_datetime(d["time"])
    tmax = pd.Series(d["temperature_2m_max"], index=idx, dtype=float)
    tmin = pd.Series(d["temperature_2m_min"], index=idx, dtype=float)
    return ((tmax + tmin) / 2).rename("temp_mid")          # 최고·최저 평균(정의된 일평균 아님)

def parse_openmeteo_pm25(obj):
    h = obj["hourly"]; s = pd.Series(h["pm2_5"], index=pd.to_datetime(h["time"]), dtype=float)
    return s.resample("D").mean().rename("pm2_5")

def parse_daily_min_temp(raw):
    df = pd.read_csv(io.BytesIO(raw)); df.columns = [c.strip().lower() for c in df.columns]
    s = pd.Series(pd.to_numeric(df["temp"], errors="coerce").values, index=pd.to_datetime(df["date"]), name="temp_mid")
    if len(s) < 3000 or not s.dropna().between(-10, 40).all():
        raise ValueError("일 최저기온 길이/범위 이상")
    return s

def synthetic_temp(n=3652, seed=SEED):
    rng = np.random.default_rng(seed); t = np.arange(n)
    return pd.Series(12 + 13 * np.sin(2 * np.pi * (t - 110) / 365.25) + rng.normal(0, 3, n) + 0.0004 * t,
                     index=pd.date_range("2015-01-01", periods=n, freq="D"), name="temp_mid")

def load_energy(which="temp"):
    temp, pm, kind = None, None, "SYNTHETIC (NOT real data)"
    if USE_LIVE_DATA:
        try:
            u = ("https://archive-api.open-meteo.com/v1/archive?latitude=37.57&longitude=126.98"
                 "&start_date=2015-01-01&end_date=2024-12-31&daily=temperature_2m_max,temperature_2m_min&timezone=Asia%2FSeoul")
            temp = parse_openmeteo_daily(json.loads(fetch_bytes(u, timeout=120)))
            kind = "REAL: Open-Meteo/ERA5 재분석 서울 일 기온 (관측소 실측 아님)"
        except Exception as e:
            print("Open-Meteo 기온 수집 실패 -> 2순위 시도:", type(e).__name__, str(e)[:120])
            try:
                temp = parse_daily_min_temp(fetch_bytes("https://raw.githubusercontent.com/jbrownlee/Datasets/master/daily-min-temperatures.csv", timeout=60))
                kind = "REAL: Melbourne 일 최저기온 1981-1990 (jbrownlee/Datasets; 서울 아님, 일평균 아닌 최저기온)"
            except Exception as e2:
                print("일 최저기온 수집 실패 -> 합성 대체:", type(e2).__name__, str(e2)[:100]); temp = None
        try:
            u = ("https://air-quality-api.open-meteo.com/v1/air-quality?latitude=37.57&longitude=126.98"
                 "&hourly=pm2_5&start_date=2022-09-01&end_date=2024-12-31&timezone=Asia%2FSeoul")
            pm = parse_openmeteo_pm25(json.loads(fetch_bytes(u, timeout=120)))
        except Exception as e:
            print("Open-Meteo 대기질 수집 실패(선택 항목):", type(e).__name__, str(e)[:100])
    if temp is None:
        temp = synthetic_temp()
    temp = temp.asfreq("D").interpolate()
    if which == "demand":
        rng = np.random.default_rng(SEED + 5); t = np.arange(len(temp))
        dow = temp.index.dayofweek.to_numpy()
        dem = 60 + 0.9 * np.abs(temp.to_numpy() - 18) ** 1.15 + 6 * (dow < 5) + 0.004 * t + rng.normal(0, 2.0, len(temp))
        return dict(y=pd.Series(dem, index=temp.index, name="demand"), X=temp.to_frame("temp"), m=7, freq="D",
                    label="일별 전력수요(합성, 기온 의존)", unit="GWh",
                    kind="SYNTHETIC-DERIVED (NOT real data): 수요는 합성, 기온 입력은 " + kind)
    return dict(y=temp, pm=pm, m=365, freq="D", label="일평균 기온", unit="°C", kind=kind)

D = load_energy("temp")
y, DATA_KIND, FREQ = D["y"], D["kind"], D["freq"]
M = D["m"]
print("DATA_KIND:", DATA_KIND); print("관측 수:", len(y), "| 기간:", y.index.min().date(), "~", y.index.max().date())

## 2. 시각화와 진단

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
y.plot(ax=ax, title=f"{D['label']} | {DATA_KIND}")
ax.set_ylabel(D["unit"]); plt.tight_layout(); plt.show()
print(y.describe().round(3).to_string())
n_missing = int(y.isna().sum()); print("결측:", n_missing, "| 추정 주기:", pd.infer_freq(y.index))

In [ ]:
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.stats.diagnostic import acorr_ljungbox, het_arch
from statsmodels.tsa.seasonal import seasonal_decompose

def stationarity_row(s, label):
    x = pd.Series(s).dropna().to_numpy(dtype=float)
    adf_p = adfuller(x, autolag="AIC")[1]
    kpss_p = kpss(x, regression="c", nlags="auto")[1]
    verdict = ("정상성과 부합" if (adf_p < 0.05 and kpss_p >= 0.05) else
               "비정상 의심(차분 고려)" if (adf_p >= 0.05 and kpss_p < 0.05) else
               "두 검정 불일치/검정력 한계: 판단 보류")
    return {"series": label, "ADF_p": adf_p, "KPSS_p": kpss_p, "읽기": verdict}

In [ ]:
# 계절성 분해 (주기 M=M). 표본이 주기의 2배 미만이면 건너뛴다.
if 2 * M <= len(y) and M > 1:
    dec = seasonal_decompose(y.astype(float), model="additive", period=M)
    parts = pd.concat([dec.observed, dec.trend, dec.seasonal, dec.resid], axis=1,
                      keys=["observed", "trend", "seasonal", "resid"])
    parts.iloc[-min(len(parts), 6 * M):].plot(subplots=True, figsize=(11, 7), title="분해 결과(최근 구간)")
    plt.tight_layout(); plt.show()
    r = dec.resid.dropna(); sr = (dec.seasonal + dec.resid).dropna()
    Fs = max(0.0, 1 - float(r.var()) / float(sr.var()))
    print(f"계절성 세기 F_S = {Fs:.3f}  (0=없음, 1=매우 강함; 휴리스틱 지표)")
else:
    Fs = np.nan; print("표본이 짧아 분해를 건너뜀")

In [ ]:
rows = [stationarity_row(y, "level")]
yd = y.diff().dropna(); rows.append(stationarity_row(yd, "1차 차분"))
if 3 * M < len(y) and M > 1 and M <= 60:
    rows.append(stationarity_row(y.diff(M).dropna(), f"계절차분(M={M})"))
res = pd.DataFrame(rows); show(res)
work = yd if rows[0]["읽기"] != "정상성과 부합" else y      # 이후 ACF 분석 대상
print("ACF 분석 대상:", "1차 차분" if work is yd else "level")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
nl = int(min(40, len(work) // 4))
plot_acf(work, lags=nl, ax=ax[0], title="ACF"); plot_pacf(work, lags=nl, ax=ax[1], title="PACF", method="ywm")
plt.tight_layout(); plt.show()
lb = acorr_ljungbox(work, lags=[10], return_df=True)
print("Ljung-Box(10) p =", round(float(lb["lb_pvalue"].iloc[0]), 4), "-> 작으면 자기상관 존재")

In [ ]:
# 분산 안정성: 앞·뒤 절반의 분산 비교(Levene)와 이동 표준편차
h = len(work) // 2
lev = stats.levene(work.iloc[:h], work.iloc[h:])
print(f"Levene p = {lev.pvalue:.4f} (작으면 전·후반 분산이 다름)")
work.rolling(max(7, M if M < 60 else 30)).std().plot(title="이동 표준편차", figsize=(11, 3.5)); plt.show()

In [ ]:
# (선택) 대기질: PM2.5 월별 기후값
pm = D.get("pm")
if pm is not None and len(pm.dropna()) > 200:
    pm.dropna().groupby(pm.dropna().index.month).mean().plot(kind="bar", title="PM2.5 월별 평균(Open-Meteo 대기질 모델값, 실측 아님)")
    plt.show()
else:
    print("PM2.5 자료가 없어 기온만 분석합니다(오프라인이거나 수집 실패).")

In [ ]:
print("=== 자동 요약(숫자 확인용, 해석은 직접 작성) ===")
print("데이터 종류:", DATA_KIND)
for r_ in res.to_dict("records"):
    print(f"- {r_['series']}: ADF p={r_['ADF_p']:.3f}, KPSS p={r_['KPSS_p']:.3f} -> {r_['읽기']}")
if not np.isnan(Fs): print(f"- 계절성 세기 F_S={Fs:.2f}")

## 3. 결과 해석 작성

1. `DATA_KIND`가 **REAL**인지 **SYNTHETIC**인지 먼저 적는다. 합성자료의 결과는 방법 시연이며 현상에 대한 증거가 아니다.
2. 수준과 차분(또는 수익률)에서 ADF·KPSS가 어떻게 다른지 설명한다. 두 검정이 어긋나면 구조변화·추세·표본기간을 논의한다.
3. 계절성 세기와 ACF/PACF가 이후 모형(Ch.2)의 후보 차수에 주는 시사점을 적는다.
4. 한계: 단일 표본, 검정력, 데이터 정의(재분석·집계·취소 제외 등).

## 4. 심화 과제
- 표본기간을 절반으로 나누어 결론이 유지되는지 확인한다.
- 차분 대신 로그변환 후 차분으로 바꿔 결과를 비교한다.
- **제출:** 그래프 3개 이상, 검정표, 해석 단락, 데이터 출처·한계.